### Modelo Bagging con K-Nearest Neighbors (KNN)

En esta sección se implementa un modelo basado en la técnica de **bagging (Bootstrap Aggregating)** utilizando como estimador base el algoritmo K-Nearest Neighbors (KNN). A diferencia de los modelos de árbol utilizados previamente, este enfoque combina múltiples modelos entrenados sobre diferentes subconjuntos del dataset, con el objetivo de reducir la varianza y mejorar la estabilidad del modelo.

El uso de KNN introduce un enfoque basado en la proximidad entre observaciones, lo que permite capturar relaciones locales en los datos. Al integrarlo dentro de un esquema de bagging, se busca mejorar su capacidad de generalización y compararlo con otros modelos del proyecto, como Decision Tree y Random Forest.


In [2]:

# === Imports generales ===
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import importlib

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# === Configurar paths ===
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# === Auto-import preprocessing modules ===
from importlib import import_module

preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"

for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("Loaded preprocessing:", ", ".join(preprocessing_modules.keys()))

Loaded preprocessing: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural


In [30]:
newexist_option = "B"
createjob_option = "A"
retainedjob_option = "A"
approvaldate_option = "A"
approvalfy_option = "B"
franchise_option = "raw"
urbanrural_option = "onehot"
revlinecr_option = "C"
lowdoc_option = "C"
disbursementgross_option = "B"

In [31]:
# === Load dataset ===
df = pd.read_csv(project_root / "data" / "train.csv")

print("Shape:", df.shape)
df.head()

Shape: (20768, 21)


,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


In [32]:
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)

noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option="log")

city_bank = importlib.reload(city_bank)
df = city_bank.get_city_bank_encoder(df)

newExists = importlib.reload(newExists)
df = newExists.preprocess_newexist(df, option=newexist_option, source_col="NewExist")

createJob = importlib.reload(createJob)
df = createJob.preprocess_createjob(df, option=createjob_option, source_col="CreateJob")

retainedJob = importlib.reload(retainedJob)
df = retainedJob.preprocess_retainedjob(df, option=retainedjob_option, source_col="RetainedJob")

approvalDate = importlib.reload(approvalDate)
df = approvalDate.preprocess_approvaldate(df, option=approvaldate_option, source_col="ApprovalDate")

approvalFY = importlib.reload(approvalFY)
df = approvalFY.preprocess_approvalfy(df, option=approvalfy_option, source_col="ApprovalFY")

franchise_code = importlib.reload(franchise_code)
df = franchise_code.preprocess_franchise_code(df, option=franchise_option, source_col="FranchiseCode")

urban_rural = importlib.reload(urban_rural)
df = urban_rural.preprocess_urban_rural(df, option=urbanrural_option, source_col="UrbanRural")

RevLineCr = importlib.reload(RevLineCr)
df = RevLineCr.preprocess_revlinecr(df, option=revlinecr_option, source_col="RevLineCr")

LowDoc = importlib.reload(LowDoc)
df = LowDoc.preprocess_lowdoc(df, option=lowdoc_option, source_col="LowDoc")

disbursementDate = importlib.reload(disbursementDate)
df = disbursementDate.preprocess_disbursementdate(df, source_col="DisbursementDate")

balanceGross = importlib.reload(balanceGross)
df = balanceGross.preprocess_balancegross(df, source_col="BalanceGross")

accept = importlib.reload(accept)
df = accept.preprocess_accept(df, source_col="Accept")

disbursementGross = importlib.reload(disbursementGross)
df = disbursementGross.preprocess_disbursementgross(
    df=df,
    option=disbursementgross_option,
    source_col="DisbursementGross",
)

print("Final shape:", df.shape)

Final shape: (20765, 40)


In [33]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# === Split ===
X = df.drop(columns=["Accept"])
y = df["Accept"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  #importante si clases desbalanceadas
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)



def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    f1_macro= f1_score(y_test, y_pred, average= "macro", zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\n=== {name} ===")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1-score :", round(f1, 4))
    print("F1_score (MACRO) :", round(f1_macro, 4))
    print("ROC-AUC  :", round(roc, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f1(macro)": f1_macro,
        "roc_auc": roc
    }

Train shape: (16612, 39)
Test shape: (4153, 39)


### Entrenamiento del modelo Bagging con KNN

En esta celda se entrena un modelo de **BaggingClassifier** utilizando como estimador base el algoritmo K-Nearest Neighbors (KNN). A diferencia de modelos individuales como Decision Tree, el bagging combina múltiples modelos entrenados sobre subconjuntos de datos, lo que permite reducir la varianza y mejorar la estabilidad del modelo.

Este enfoque busca aprovechar la simplicidad de KNN junto con la robustez del ensemble, obteniendo un modelo más generalizable frente a los datos.


In [34]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

bagging_knn = BaggingClassifier(
    estimator=KNeighborsClassifier(n_neighbors=5),
    n_estimators=50,
    random_state=42,
    n_jobs=1
)

bagging_knn.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",KNeighborsClassifier()
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",50
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary `... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


### Evaluación del modelo Bagging con KNN

En esta celda se evalúa el rendimiento del modelo Bagging con KNN sobre el conjunto de test. Se calculan diversas métricas de clasificación, como accuracy, precision, recall, F1-score, macro F1-score y ROC-AUC, que permiten analizar tanto el rendimiento global como el equilibrio entre clases.

Los resultados obtenidos se utilizarán posteriormente para comparar este modelo con otros enfoques, como Decision Tree y Random Forest.


In [35]:
results_bagging_knn = evaluate_model(
    bagging_knn,
    X_test,
    y_test,
    name="Bagging + KNN"
)

results_bagging_knn


=== Bagging + KNN ===
Accuracy : 0.7818
Precision: 0.8275
Recall   : 0.906
F1-score : 0.865
F1_score (MACRO) : 0.6487
ROC-AUC  : 0.7443

Classification Report:
              precision    recall  f1-score   support

         0.0       0.53      0.36      0.43       950
         1.0       0.83      0.91      0.86      3203

    accuracy                           0.78      4153
   macro avg       0.68      0.63      0.65      4153
weighted avg       0.76      0.78      0.77      4153



{'accuracy': 0.7818444497953286,
 'precision': 0.827487881380097,
 'recall': 0.9060256009990634,
 'f1': 0.8649776453055141,
 'f1(macro)': 0.6486542361865917,
 'roc_auc': 0.7443084608179832}


### Ajuste de hiperparámetros del modelo Bagging con KNN

En esta celda se optimizan los principales hiperparámetros del modelo Bagging con KNN mediante GridSearchCV. Se exploran diferentes configuraciones tanto del clasificador base KNN, como el número de vecinos o el tipo de ponderación, como del propio ensemble de bagging, incluyendo el número de estimadores y la proporción de muestras y variables utilizadas en cada submodelo.

El objetivo es identificar la combinación que maximiza el macro F1-score, favoreciendo un comportamiento más equilibrado entre clases.

In [36]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

bagging_knn = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "estimator__n_neighbors": [3, 5],
    "estimator__weights": ["uniform", "distance"],
    "n_estimators": [10, 30],
    "max_samples": [0.8, 1.0],
    "max_features": [0.8, 1.0]
}

grid_search_bagging_knn = GridSearchCV(
    estimator=bagging_knn,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search_bagging_knn.fit(X_train, y_train)

best_bagging_knn = grid_search_bagging_knn.best_estimator_

print("Mejores parámetros:")
print(grid_search_bagging_knn.best_params_)

tuned_results_bagging_knn = evaluate_model(
    best_bagging_knn,
    X_test,
    y_test,
    name="Bagging + KNN Tuned"
)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Mejores parámetros:
{'estimator__n_neighbors': 5, 'estimator__weights': 'uniform', 'max_features': 1.0, 'max_samples': 0.8, 'n_estimators': 30}

=== Bagging + KNN Tuned ===
Accuracy : 0.7833
Precision: 0.8265
Recall   : 0.9101
F1-score : 0.8663
F1_score (MACRO) : 0.6476
ROC-AUC  : 0.7478

Classification Report:
              precision    recall  f1-score   support

         0.0       0.54      0.36      0.43       950
         1.0       0.83      0.91      0.87      3203

    accuracy                           0.78      4153
   macro avg       0.68      0.63      0.65      4153
weighted avg       0.76      0.78      0.77      4153



### Análisis de resultados del modelo Bagging con KNN

Los resultados obtenidos muestran un rendimiento muy similar entre el modelo base y el modelo optimizado mediante tuning. En concreto, el modelo ajustado presenta una ligera mejora en métricas como **accuracy (0.7833), recall (0.9101), F1-score (0.8663)** y **ROC-AUC (0.7478)**, lo que indica una mejora marginal en su capacidad predictiva.

Sin embargo, el **F1-score macro** se mantiene prácticamente constante (en torno a 0.648), lo que sugiere que el modelo sigue teniendo dificultades para equilibrar correctamente el rendimiento entre clases, especialmente en la clase minoritaria.

En conjunto, estos resultados reflejan que el proceso de optimización no produce una mejora significativa respecto al modelo base, lo que indica una cierta estabilidad del modelo frente a cambios en los hiperparámetros. No obstante, el rendimiento global sigue siendo inferior al obtenido por modelos ensemble basados en árboles, como Random Forest, especialmente en términos de equilibrio entre clases.


In [37]:
comparison_bagging_knn = pd.DataFrame([
    {"model": "Bagging + KNN Base", **results_bagging_knn},
    {"model": "Bagging + KNN Tuned", **tuned_results_bagging_knn}
])

comparison_bagging_knn

,model,accuracy,precision,recall,f1,f1(macro),roc_auc
0,Bagging + KNN Base,0.781844,0.827488,0.906026,0.864978,0.648654,0.744308
1,Bagging + KNN Tuned,0.783289,0.826481,0.910084,0.866270,0.647602,0.747803


---
## Modelo Bagging con Regresión Logística (Optimización Avanzada)
**Responsable:** Ivan Angeles

A continuación, implementaremos un segundo enfoque para el ensamblaje de Bagging, utilizando **Regresión Logística** como estimador base. 

Dado que los modelos lineales son altamente sensibles a las diferentes escalas de las variables, incorporaremos un paso obligatorio de **Estandarización (`StandardScaler`)**. Además, para llevar la optimización al máximo nivel, ejecutaremos un "Torneo" automatizado que probará las **64 combinaciones posibles** del *Feature Engineering* (Opciones A y B) para que el algoritmo seleccione empíricamente la base de datos que maximice su rendimiento, finalizando con un `GridSearchCV` para sus hiperparámetros.

In [3]:
import itertools
import pandas as pd
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 1. Definimos la ruta principal del proyecto automáticamente
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# 2. Configuramos el Torneo
opciones = {
    'newExists': ['A', 'B'],
    'createJob': ['A', 'B'],
    'retainedJob': ['A', 'B'],
    'approvalDate': ['A', 'B'],
    'approvalFY': ['A', 'B'],
    'disbursementGross': ['A', 'B']
}

keys = list(opciones.keys())
combinaciones = list(itertools.product(*(opciones[k] for k in keys)))

mejor_f1_lr = 0
mejor_combo_lr = None
mejor_df_lr = None

# 3. Cargamos el dataset utilizando la ruta correcta
df_raw = pd.read_csv(project_root / "data" / "train.csv")

print("⏳ Iniciando torneo de 64 combinaciones A/B para Bagging + Logistic Regression...")

# 4. Ciclo del Torneo
for combo in combinaciones:
    df_temp = df_raw.copy()
    
    # Limpieza Base (Heredada del Feature Engineering)
    df_temp = base_cleaning.clean_base_columns(df_temp)
    df_temp = city_bank.get_city_bank_encoder(df_temp)
    df_temp = noemp.preprocess_noemp(df_temp, option="log")
    df_temp = franchise_code.preprocess_franchise_code(df_temp, option="binary", source_col="FranchiseCode")
    df_temp = urban_rural.preprocess_urban_rural(df_temp, option="onehot", source_col="UrbanRural")
    df_temp = RevLineCr.preprocess_revlinecr(df_temp, option="C", source_col="RevLineCr")
    df_temp = LowDoc.preprocess_lowdoc(df_temp, option="C", source_col="LowDoc")
    df_temp = disbursementDate.preprocess_disbursementdate(df_temp, source_col="DisbursementDate")
    df_temp = balanceGross.preprocess_balancegross(df_temp, source_col="BalanceGross")
    df_temp = accept.preprocess_accept(df_temp, source_col="Accept")
    
    # Variables Dinámicas del Torneo
    df_temp = newExists.preprocess_newexist(df_temp, option=combo[0], source_col="NewExist")
    df_temp = createJob.preprocess_createjob(df_temp, option=combo[1], source_col="CreateJob")
    df_temp = retainedJob.preprocess_retainedjob(df_temp, option=combo[2], source_col="RetainedJob")
    df_temp = approvalDate.preprocess_approvaldate(df_temp, option=combo[3], source_col="ApprovalDate")
    df_temp = approvalFY.preprocess_approvalfy(df_temp, option=combo[4], source_col="ApprovalFY")
    df_temp = disbursementGross.preprocess_disbursementgross(df_temp, option=combo[5], source_col="DisbursementGross")
    
    # Filtro Numérico Estricto
    df_temp = df_temp.select_dtypes(include=['int16', 'int32', 'int64', 'float16', 'float32', 'float64', 'bool', 'uint8', 'Int64', 'Float64'])
    
    X = df_temp.drop(columns=['Accept'])
    y = df_temp['Accept']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # 🚨 ESTANDARIZACIÓN OBLIGATORIA PARA REGRESIÓN LOGÍSTICA 🚨
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Entrenamos Bagging + LogReg (con class_weight para balancear)
    base_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    modelo_temp = BaggingClassifier(estimator=base_lr, n_estimators=10, random_state=42, n_jobs=-1)
    
    modelo_temp.fit(X_train_scaled, y_train)
    y_pred = modelo_temp.predict(X_test_scaled)
    
    f1 = f1_score(y_test, y_pred, average='macro')
    
    if f1 > mejor_f1_lr:
        mejor_f1_lr = f1
        mejor_combo_lr = combo
        mejor_df_lr = df_temp

print(f"\n🏆 ¡Torneo finalizado! Combinación ganadora (LogReg): {dict(zip(keys, mejor_combo_lr))}")
print(f"📊 F1-Score Base obtenido: {mejor_f1_lr:.4f}")

⏳ Iniciando torneo de 64 combinaciones A/B para Bagging + Logistic Regression...

🏆 ¡Torneo finalizado! Combinación ganadora (LogReg): {'newExists': 'B', 'createJob': 'A', 'retainedJob': 'A', 'approvalDate': 'A', 'approvalFY': 'A', 'disbursementGross': 'A'}
📊 F1-Score Base obtenido: 0.6271


### Hyperparameter Tuning para Bagging (Logistic Regression)
Una vez hallada la combinación de datos óptima para este modelo lineal, aplicamos `GridSearchCV` para ajustar la arquitectura del Bagging.

In [4]:
print("⏳ Iniciando GridSearch para Bagging + Logistic Regression...")

# Split de los datos ganadores
X_best = mejor_df_lr.drop(columns=['Accept'])
y_best = mejor_df_lr['Accept']
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_best, y_best, test_size=0.2, random_state=42, stratify=y_best)

# Estandarización Final
scaler_final = StandardScaler()
X_train_b_scaled = scaler_final.fit_transform(X_train_b)
X_test_b_scaled = scaler_final.transform(X_test_b)

# Configuración del GridSearch
base_lr_tuned = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
bagging_base = BaggingClassifier(estimator=base_lr_tuned, random_state=42)

param_grid_bagging = {
    'n_estimators': [10, 50],
    'max_samples': [0.5, 0.8, 1.0],
    'max_features': [0.5, 0.8, 1.0]
}

grid_search_lr = GridSearchCV(estimator=bagging_base, param_grid=param_grid_bagging, scoring="f1_macro", cv=3, n_jobs=-1)
grid_search_lr.fit(X_train_b_scaled, y_train_b)
best_bagging_lr = grid_search_lr.best_estimator_

# Evaluación Final
y_pred_lr = best_bagging_lr.predict(X_test_b_scaled)
y_prob_lr = best_bagging_lr.predict_proba(X_test_b_scaled)[:, 1]

print("\n=== Bagging + Logistic Regression Tuned ===")
print(f"Mejores Hiperparámetros: {grid_search_lr.best_params_}")
print(f"Accuracy         : {accuracy_score(y_test_b, y_pred_lr):.4f}")
print(f"Precision (Macro): {precision_score(y_test_b, y_pred_lr, average='macro'):.4f}")
print(f"Recall (Macro)   : {recall_score(y_test_b, y_pred_lr, average='macro'):.4f}")
print(f"F1-score (Macro) : {f1_score(y_test_b, y_pred_lr, average='macro'):.4f}")
print(f"ROC-AUC          : {roc_auc_score(y_test_b, y_prob_lr):.4f}\n")
print("Classification Report:")
print(classification_report(y_test_b, y_pred_lr))

⏳ Iniciando GridSearch para Bagging + Logistic Regression...

=== Bagging + Logistic Regression Tuned ===
Mejores Hiperparámetros: {'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 10}
Accuracy         : 0.6696
Precision (Macro): 0.6316
Recall (Macro)   : 0.6822
F1-score (Macro) : 0.6244
ROC-AUC          : 0.7299

Classification Report:
              precision    recall  f1-score   support

         0.0       0.38      0.71      0.49       950
         1.0       0.88      0.66      0.75      3203

    accuracy                           0.67      4153
   macro avg       0.63      0.68      0.62      4153
weighted avg       0.77      0.67      0.70      4153



## Conclusión Individual: Modelo Bagging con Regresión Logística

**¿Qué se hizo?**
Para este enfoque, construimos un modelo de ensamblaje (*Bagging Classifier*) utilizando la **Regresión Logística** como estimador base. Dado que los modelos lineales son sumamente sensibles a las diferentes escalas de las variables, implementamos un paso riguroso de estandarización (`StandardScaler`). Además, diseñamos un "Torneo" automatizado que evaluó 64 combinaciones de *Feature Engineering* para que el algoritmo seleccionara empíricamente su base de datos óptima, finalizando con una optimización exhaustiva de hiperparámetros mediante `GridSearchCV`.

**Resultados Detallados:**
* **Configuración Óptima:** El modelo alcanzó su mejor versión ensamblando 10 estimadores (`n_estimators=10`), entrenando cada uno con el 50% de las muestras (`max_samples=0.5`) y utilizando el 100% de las variables (`max_features=1.0`), lo cual inyecta la varianza necesaria para evitar el sobreajuste.
* **El Valor para el Negocio (Clase Minoritaria):** Aunque el modelo obtuvo un *F1-Score Macro* de 0.6244 y un *Accuracy* del 67%, su verdadero triunfo radica en la detección del riesgo. Gracias al balanceo de clases automático (`class_weight='balanced'`), el modelo logró un **Recall del 71% en los préstamos rechazados (clase 0.0)**. Esto significa que es capaz de detectar 7 de cada 10 préstamos que van a fracasar, una métrica excepcionalmente valiosa para evitar pérdidas financieras en el banco.

## Conclusión Global: Modelos de Ensamblaje (Bagging KNN vs. Bagging Regresión Logística)

**El Veredicto Final del Equipo de Modelos Bagging:**
Al evaluar las dos estrategias base (K-Nearest Neighbors y Regresión Logística) dentro del esquema de Bagging, observamos contrastes muy interesantes derivados de la naturaleza matemática de cada algoritmo:

1. **Bagging + KNN:** El modelo basado en distancias logró un *F1-Score Macro* ligeramente superior (**0.648**). Es un modelo que encuentra patrones locales de forma eficiente, pero sufre con la "maldición de la dimensionalidad" tras haber codificado decenas de ciudades y bancos, y le cuesta generalizar la clase minoritaria sin un ajuste manual muy complejo.

2. **Bagging + Regresión Logística:** Aunque su métrica global es ligeramente menor, este enfoque solucionó el problema fundamental de la base de datos: **el desbalance**. Mientras KNN busca similitudes, la Regresión Logística penaliza matemáticamente los errores. Esto permitió disparar la sensibilidad del modelo para detectar la clase minoritaria.

**Recomendación:** Si el objetivo principal de la institución es tener un modelo con un rendimiento global estable, el **Bagging con KNN** es una opción sólida. Sin embargo, si la prioridad de negocio del banco es la **gestión de riesgo y evitar pérdidas de capital**, el modelo **Bagging con Regresión Logística** es muy superior, ya que su capacidad para "levantar la bandera roja" ante un mal préstamo es mucho más precisa y confiable.